# Notebook 02 — Silver · Limpeza e Padronização

**Objetivo:** Aplicar regras de qualidade sobre as tabelas Bronze, gerando versões limpas e tipadas.  
**Fonte:** Schema `bronze`  
**Destino:** Schema `silver`  
**Autor:** Davi Alves  

### Problemas identificados na Bronze
| Campo | Problema | Exemplo |
|---|---|---|
| Status | 8 variações de capitalização | `conciliado`, `CONCILIADO`, `Conciliado` |
| Tipo_Lancamento | 5 variações | `pagamento`, `PAGAMENTO`, `Pagamento` |
| Canal | 9 variações | `PIX`, `Pix`, `pix`, `TED`, `ted` |
| Valor_Previsto / Realizado | Vírgula como separador decimal | `"32410,99"` |
| Datas | Tipo string — precisam virar DateType | `2024-05-06` |

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.silver")
print("Schema silver criado com sucesso")

Schema silver criado com sucesso


In [0]:
# Lê as tabelas Bronze como ponto de partida
df_transacoes   = spark.table("workspace.bronze.transacoes")
df_contas       = spark.table("workspace.bronze.contas")
df_fornecedores = spark.table("workspace.bronze.fornecedores")

print(f"Bronze carregado — transacoes: {df_transacoes.count()} linhas")
print(f"Bronze carregado — contas: {df_contas.count()} linhas")
print(f"Bronze carregado — fornecedores: {df_fornecedores.count()} linhas")

Bronze carregado — transacoes: 3030 linhas
Bronze carregado — contas: 8 linhas
Bronze carregado — fornecedores: 10 linhas


In [0]:
from pyspark.sql.functions import col, trim, initcap, regexp_replace, to_date, when
from pyspark.sql.types import DecimalType

# Corrige vírgula decimal e converte para número
def fix_decimal(column):
    return regexp_replace(col(column), ",", ".").cast(DecimalType(15, 2))

df_silver_transacoes = df_transacoes \
    .withColumn("Status",           initcap(trim(col("Status")))) \
    .withColumn("Tipo_Lancamento",  initcap(trim(col("Tipo_Lancamento")))) \
    .withColumn("Canal",            initcap(trim(col("Canal")))) \
    .withColumn("Valor_Previsto",   fix_decimal("Valor_Previsto")) \
    .withColumn("Valor_Realizado",  fix_decimal("Valor_Realizado")) \
    .withColumn("Data_Lancamento",  to_date(col("Data_Lancamento"),  "yyyy-MM-dd")) \
    .withColumn("Data_Prevista",    to_date(col("Data_Prevista"),    "yyyy-MM-dd")) \
    .withColumn("Data_Liquidacao",  to_date(col("Data_Liquidacao"),  "yyyy-MM-dd")) \
    .drop("dt_carga")

print("Limpeza aplicada com sucesso")
print(f"Linhas: {df_silver_transacoes.count()}")
print("\nSchema resultante:")
df_silver_transacoes.printSchema()

Limpeza aplicada com sucesso
Linhas: 3030

Schema resultante:
root
 |-- ID_Transacao: string (nullable = true)
 |-- Data_Lancamento: date (nullable = true)
 |-- Data_Prevista: date (nullable = true)
 |-- Data_Liquidacao: date (nullable = true)
 |-- Valor_Previsto: decimal(15,2) (nullable = true)
 |-- Valor_Realizado: decimal(15,2) (nullable = true)
 |-- Status: string (nullable = true)
 |-- Tipo_Lancamento: string (nullable = true)
 |-- ID_Conta: string (nullable = true)
 |-- ID_Fornecedor: string (nullable = true)
 |-- Descricao: string (nullable = true)
 |-- Canal: string (nullable = true)



In [0]:
print("=== Validação dos valores únicos após limpeza ===\n")

print("Status:")
df_silver_transacoes.groupBy("Status").count().orderBy("Status").show()

print("Tipo_Lancamento:")
df_silver_transacoes.groupBy("Tipo_Lancamento").count().orderBy("Tipo_Lancamento").show()

print("Canal:")
df_silver_transacoes.groupBy("Canal").count().orderBy("Canal").show()

=== Validação dos valores únicos após limpeza ===

Status:
+----------+-----+
|    Status|count|
+----------+-----+
|Conciliado| 2395|
|Divergente|  314|
|  Pendente|  321|
+----------+-----+

Tipo_Lancamento:
+---------------+-----+
|Tipo_Lancamento|count|
+---------------+-----+
|        Estorno|  556|
|      Pagamento| 1857|
|    Recebimento|  617|
+---------------+-----+

Canal:
+------+-----+
| Canal|count|
+------+-----+
|Boleto|  682|
|Debito|  352|
|Débito|  322|
|   Pix| 1023|
|   Ted|  651|
+------+-----+



In [0]:
df_silver_transacoes = df_silver_transacoes.withColumn(
    "Canal",
    when(col("Canal") == "Debito", "Débito").otherwise(col("Canal"))
)

print("Validação após correção do acento:")
df_silver_transacoes.groupBy("Canal").count().orderBy("Canal").show()

Validação após correção do acento:
+------+-----+
| Canal|count|
+------+-----+
|Boleto|  682|
|Débito|  674|
|   Pix| 1023|
|   Ted|  651|
+------+-----+



In [0]:
# Salva transacoes limpa
df_silver_transacoes.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.transacoes")

# Contas e fornecedores só precisam de tipagem — remove dt_carga e salva
df_contas.drop("dt_carga").write.format("delta").mode("overwrite").saveAsTable("workspace.silver.contas")
df_fornecedores.drop("dt_carga").write.format("delta").mode("overwrite").saveAsTable("workspace.silver.fornecedores")

print("Silver salvo com sucesso")
print(f"silver.transacoes:   {spark.table('workspace.silver.transacoes').count()} linhas")
print(f"silver.contas:       {spark.table('workspace.silver.contas').count()} linhas")
print(f"silver.fornecedores: {spark.table('workspace.silver.fornecedores').count()} linhas")

Silver salvo com sucesso
silver.transacoes:   3030 linhas
silver.contas:       8 linhas
silver.fornecedores: 10 linhas


## Relatório de qualidade — Silver

### Transformações aplicadas em silver.transacoes

| Campo | Problema original | Solução aplicada | Resultado |
|---|---|---|---|
| Status | 8 variações de capitalização | `initcap` + `trim` | 3 valores: Conciliado, Pendente, Divergente |
| Tipo_Lancamento | 5 variações | `initcap` + `trim` | 3 valores: Pagamento, Recebimento, Estorno |
| Canal | 9 variações + acento | `initcap` + correção manual | 4 valores: Pix, Ted, Boleto, Débito |
| Valor_Previsto / Realizado | Vírgula decimal em 2.210 linhas | `regexp_replace` + cast Decimal(15,2) | Tipo numérico correto |
| Datas | Tipo string | `to_date` formato yyyy-MM-dd | Tipo DateType |

### Contagem final

| Tabela | Linhas | Status |
|---|---|---|
| silver.transacoes | 3.030 | ✅ Limpo |
| silver.contas | 8 | ✅ Limpo |
| silver.fornecedores | 10 | ✅ Limpo |

### Observação sobre nulos
`Data_Liquidacao` e `Valor_Realizado` nulos são **esperados** para lançamentos com Status = Pendente.  
Não foram removidos — fazem parte da regra de negócio da conciliação.

In [0]:
# Exporta dContas e dFornecedores Silver como CSV
spark.table("workspace.silver.contas").coalesce(1).write \
    .option("header", True).mode("overwrite") \
    .csv("/Volumes/workspace/conciliacao/raw_data/silver_contas")

spark.table("workspace.silver.fornecedores").coalesce(1).write \
    .option("header", True).mode("overwrite") \
    .csv("/Volumes/workspace/conciliacao/raw_data/silver_fornecedores")

print("Exportação concluída")
print(f"silver.contas:       {spark.table('workspace.silver.contas').count()} linhas")
print(f"silver.fornecedores: {spark.table('workspace.silver.fornecedores').count()} linhas")

Exportação concluída
silver.contas:       8 linhas
silver.fornecedores: 10 linhas
